<a href="https://colab.research.google.com/github/eiyu18/MachineLearning/blob/main/JS09/JS09_Lab04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install annoy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 6.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520032 sha256=09b31058ebc948b666bbeec27a49f24326aed05094a3ba6aa70891dad254e814
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy


In [3]:
!pip install faiss-cpu
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 31.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 MB 16.8 MB/s  0:00:08
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 11.4 MB/s  0:00:47
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 18.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 21.0 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 12.0 MB/s  0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [faiss-gpu]


In [4]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679797 sha256=971d12d714bb8b58a1549b84bb5769d8f4bf01c036ad7c51ec1a2f5591298bdd
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [5]:
	import numpy as np
	import time
	from annoy import AnnoyIndex
	import faiss
	import hnswlib

	# ===============================
	# Build 1 million rows with 5 dimensions
	# ===============================
	n_data = 1_000_000   # use 100_000 if memory is constrained
	dim = 5
	X = np.random.random((n_data, dim)).astype(np.float32)

	# Query point
	query = np.random.random((1, dim)).astype(np.float32)
	k = 10

	# ===============================
	# 2. Annoy
	# ===============================
	print("=== Annoy ===")
	ann_index = AnnoyIndex(dim, 'euclidean')

	start = time.time()
	for i in range(n_data):
	    ann_index.add_item(i, X[i])
	ann_index.build(10)  # 10 trees
	build_time = time.time() - start

	start = time.time()
	neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
	query_time = time.time() - start

	print("Build time:", build_time, "seconds")
	print("Query time:", query_time, "seconds")
	print("Neighbors:", neighbors[0][:5], "...")

	# ===============================
	# 3. FAISS (Flat Index)
	# ===============================
	print("\n=== FAISS (IndexFlatL2) ===")
	faiss_index = faiss.IndexFlatL2(dim)

	start = time.time()
	faiss_index.add(X)
	build_time = time.time() - start

	start = time.time()
	distances, indices = faiss_index.search(query, k)
	query_time = time.time() - start

	print("Build time:", build_time, "seconds")
	print("Query time:", query_time, "seconds")
	print("Neighbors:", indices[0][:5], "...")

	# ===============================
	# 4. HNSW (hnswlib)
	# ===============================
	print("\n=== HNSW (hnswlib) ===")
	hnsw_index = hnswlib.Index(space='l2', dim=dim)

	start = time.time()
	hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
	hnsw_index.add_items(X)
	build_time = time.time() - start

	hnsw_index.set_ef(50)

	start = time.time()
	labels, distances = hnsw_index.knn_query(query, k=k)
	query_time = time.time() - start

	print("Build time:", build_time, "seconds")
	print("Query time:", query_time, "seconds")
	print("Neighbors:", labels[0][:5], "...")

=== Annoy ===
Build time: 33.859150409698486 seconds
Query time: 0.00037789344787597656 seconds
Neighbors: [102602, 401480, 964050, 931384, 383986] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.022983074188232422 seconds
Query time: 0.0299990177154541 seconds
Neighbors: [102602 401480 964050 931384 383986] ...

=== HNSW (hnswlib) ===
Build time: 290.9030544757843 seconds
Query time: 0.0006196498870849609 seconds
Neighbors: [102602 401480 964050 931384 383986] ...
